# Sweep Engine + Expected Value Optimization Examples

This notebook shows how to use:
- `ConditionTemplate` and `ConditionSweepEngine`
- `ExpectedValueOptimizer`
- `expected_value_from_outcomes` and `pooled_expected_value_by_group`

It uses inter-day data loaded via `strategies.data` (per repo conventions).

In [1]:
import pandas as pd

from strategies.data import add_market_data_to_syspath, load_daily_variables
from strategies.backtests import ConditionTemplate, ConditionSweepEngine
from strategies.optimization import (
    ExpectedValueOptimizer,
    expected_value_from_outcomes,
    pooled_expected_value_by_group,
)
from strategies.results import BacktestResultStore

In [2]:
# Load preprocessed inter-day data through strategies.data
add_market_data_to_syspath()
symbols = load_daily_variables(filename="daily_variables.pkl")

if not isinstance(symbols, dict) or not symbols:
    raise ValueError("Expected non-empty dict of symbol -> SymbolData")

available = sorted(symbols.keys())
selected_symbols = available[:5]  # adjust as desired

data_by_symbol = {sym: symbols[sym].df.copy() for sym in selected_symbols}
print(f"Selected symbols ({len(selected_symbols)}): {selected_symbols}")

Loading Variables: 100%|██████████| 26/26 [04:12<00:00,  9.71s/it]  


Selected symbols (5): ['A', 'AA', 'AAL', 'AAMI', 'AAOI']


In [3]:
# Discover available RSI columns like rsi_13, rsi_14, ...
sample_df = next(iter(data_by_symbol.values()))
rsi_cols = []
for col in sample_df.columns:
    c = str(col).lower()
    if c.startswith("rsi_"):
        suffix = c.split("_", 1)[1]
        if suffix.isdigit():
            rsi_cols.append((int(suffix), col))

rsi_cols = sorted(rsi_cols, key=lambda x: x[0])
if not rsi_cols:
    raise ValueError("No RSI columns like rsi_13 found. Update template to match your indicator columns.")

lengths = tuple(x[0] for x in rsi_cols[:3])  # use first 3 discovered periods
print("Using RSI lengths:", lengths)
print("Columns:", [x[1] for x in rsi_cols[:3]])

Using RSI lengths: (14,)
Columns: ['RSI_14']


In [ ]:
# Define template(s): this creates a cartesian grid of condition variants
templates = [
    ConditionTemplate(
        name="rsi_signal",
        indicator_template="rsi_{length}",
        operator=">",
        threshold_values=(60, 65, 70),
        parameter_grid={"length": lengths},
    )
]

store = BacktestResultStore("results/sweep_backtests.json")
engine = ConditionSweepEngine(result_store=store, overwrite_results=True)

condition_sets = engine.generate_condition_sets(
    templates,
    set_name_prefix="rsi_ev",
)

print(f"Generated {len(condition_sets)} condition sets")
condition_sets[:2]

In [ ]:
# Run sweep: each condition set is converted to event dates and backtested with backtrader
evaluations = engine.run_sweep(
    sweep_name="daily_rsi_expected_value",
    condition_sets=condition_sets,
    data_by_symbol=data_by_symbol,
    hold_bars=3,
    close_column="close",
    metadata={"note": "example notebook sweep"},
)

print(f"Completed evaluations: {len(evaluations)}")

In [ ]:
# Rank by expected value overall (pooled trades across symbols)
optimizer = ExpectedValueOptimizer()
top_overall = optimizer.rank(evaluations, min_sample_size=20, top_n=10)

pd.DataFrame([
    {
        "rank": r.rank,
        "set_name": r.set_name,
        "overall_ev": r.overall_expected_value,
        "overall_n": r.overall_sample_size,
        "total_trades": r.total_trades,
    }
    for r in top_overall
])

In [ ]:
# Rank by expected value for one symbol
target_symbol = selected_symbols[0]
top_for_symbol = optimizer.rank(
    evaluations,
    symbol=target_symbol,
    min_sample_size=5,
    top_n=10,
)

print("Target symbol:", target_symbol)
pd.DataFrame([
    {
        "rank": r.rank,
        "set_name": r.set_name,
        "symbol_ev": r.objective_expected_value,
        "symbol_n": r.objective_sample_size,
        "overall_ev": r.overall_expected_value,
    }
    for r in top_for_symbol
])

In [ ]:
# Direct EV helper examples
example_outcomes = [100, -50, 40, -20, 10]
single_stats = expected_value_from_outcomes(example_outcomes)

grouped = {
    "AAPL": [120, -40, 30],
    "MSFT": [50, -10, -5, 20],
}
pooled_stats = pooled_expected_value_by_group(grouped)

print("Single sample EV:", single_stats.expected_value, "n=", single_stats.sample_size)
print("Pooled EV:", pooled_stats.expected_value, "n=", pooled_stats.sample_size)